# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [4]:
%%sh
git config --global user.email "you@example.com"
git config --global user.name "Your Name"
git init
pip install scikit-learn numpy pandas -qqq
pip freeze > requirements.txt

Reinitialized existing Git repository in /content/.git/


In [5]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


### Проверяем работоспособность пайплайна

In [6]:
!python ml_pipeline.py

Точность аccuracy: 1.00


In [7]:
%%writefile .gitlab-ci.yml
stages:
  - build
  - train

build_env:
  stage: build
  image: python:3.9
  script:
    - pip install scikit-learn numpy pandas
    - pip freeze > requirements.txt
  artifacts:
    paths:
      - requirements.txt

train_ml_model:
  stage: train
  image: python:3.9
  script:
    - pip install -r requirements.txt
    - python ml_pipeline.py
  artifacts:
    paths:
      - ml_pipeline.py
    expire_in: 1 day

Writing .gitlab-ci.yml


In [12]:
!git add .gitlab-ci.yml requirements.txt ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitLab"
!git log

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)
commit 323072fa0f63e0eb49a98bd2481abf5cd7c2ba1e (HEAD -> main)
Author: Your Name <you@example.com>
Date:   Mon May 11 16:39:59 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн GitLab


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

https://gitlab.com/alexeroshkin1-group/project1/-/pipelines

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

# ADR 002: Выбор стратегии Rolling Update для ML-сервиса

**Статус:** Accepted  
**Дата:** 2026-05-11

## Контекст
Нам необходимо развертывать ML-сервис для классификации данных Iris.
Ограничения:
- Ограниченные ресурсы (один сервер/кластер).
- Требование доступности 24/7 (zero downtime).

## Решение
Мы будем использовать стратегию **Rolling Update** (постепенное обновление).
Новые инстансы сервиса с обновленной моделью будут запускаться параллельно со старыми, постепенно заменяя их.

## Последствия (Риски и выгоды)

### Плюсы (Benefits):
- **Zero Downtime**: Пользователи не заметят процесса обновления.
- **Экономия ресурсов**: Нам не нужно удваивать количество серверов, как в Blue-Green.

### Минусы и риски (Risks):
- **Смешивание версий**: В процессе деплоя часть запросов уйдет на старую модель, часть — на новую. Это критично, если формат входных данных (API) изменился.
- **Сложность отката**: Если новая модель работает некорректно, откат (Rollback) также будет происходить постепенно, что увеличивает время нахождения "плохой" версии в продакшене.
- **Нагрузка**: В момент обновления на сервер ложится дополнительная нагрузка по памяти для запуска временных контейнеров.


## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [14]:
%%writefile docker-compose-blue.yaml
version: '3.8'

services:
  ml-service:
    image: python:3.9-slim
    container_name: ml-container-blue
    restart: always
    volumes:
      - .:/app
    working_dir: /app
    command: >
      sh -c "pip install scikit-learn numpy pandas && python ml_pipeline.py"
    deploy:
      replicas: 2
      update_config:
        parallelism: 1
        delay: 10s
        order: start-first
        failure_action: rollback
    healthcheck:
      test: ["CMD", "python", "-c", "import scikit_learn"]
      interval: 30s
      timeout: 10s
      retries: 3
    networks:
      - ml-net

networks:
  ml-net:
    driver: bridge

Writing docker-compose-blue.yaml


## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

In [15]:
experiment_plan = {
    "experiment_id": "EXP_001_IRIS_REORDER",
    "description": "Сравнение базовой модели RandomForest (A) с новой версией (B)",

    "traffic_split": {
        "control_group_A": 0.5,
        "test_group_B": 0.5
    },

    "metrics": {
        "primary": "accuracy_score",
        "secondary": "latency_ms",
        "business": "click_through_rate"
    },

    "statistical_params": {
        "alpha": 0.05,
        "power": 0.8,
        "mde": 0.02
    },

    "duration": "7 days",
    "targeting": "all_users",
    "success_criteria": "Group B accuracy > Group A accuracy AND p-value < 0.05"
}

import json
print(json.dumps(experiment_plan, indent=4, ensure_ascii=False))

{
    "experiment_id": "EXP_001_IRIS_REORDER",
    "description": "Сравнение базовой модели RandomForest (A) с новой версией (B)",
    "traffic_split": {
        "control_group_A": 0.5,
        "test_group_B": 0.5
    },
    "metrics": {
        "primary": "accuracy_score",
        "secondary": "latency_ms",
        "business": "click_through_rate"
    },
    "statistical_params": {
        "alpha": 0.05,
        "power": 0.8,
        "mde": 0.02
    },
    "duration": "7 days",
    "targeting": "all_users",
    "success_criteria": "Group B accuracy > Group A accuracy AND p-value < 0.05"
}


## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [ ]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


Проверяем работоспособность пайплайна

In [ ]:
!python ml_pipeline.py

Точность аccuracy: 1.00


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [16]:
%%writefile ci.yml
name: CI
on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout code
      uses: actions/checkout@v2

    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.9'

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install scikit-learn numpy pandas joblib
        pip freeze > requirements.txt

    - name: Run pipeline
      run: |
        python ml_pipeline.py

    - name: Make pipeline reproducible
      run: |
        echo "Recording dependencies and model state..."
        mkdir -p output
        cp requirements.txt output/
        cp ml_pipeline.py output/

    - name: Upload ML Artifacts
      uses: actions/upload-artifact@v4
      with:
        name: ml-reproducibility-package
        path: output/

Writing ci.yml


Копируем ci.yml в правильную директорию .github/workflows

In [17]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml

Начинаем отправку в репозиторий

In [18]:
!git add ./.github/workflows/ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
!git log

[main 39e8a7d] build(ml_pipeline.py) добавлен пайплайн GitHub Actions
 1 file changed, 38 insertions(+)
 create mode 100644 .github/workflows/ci.yml
commit 39e8a7d1bd87dd38d298d9d123f448c8488e7e5c (HEAD -> main)
Author: Your Name <you@example.com>
Date:   Mon May 11 18:00:45 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн GitHub Actions

commit 323072fa0f63e0eb49a98bd2481abf5cd7c2ba1e
Author: Your Name <you@example.com>
Date:   Mon May 11 16:39:59 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн GitLab


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

https://github.com/Alexer777/MLOPS_HW7/actions

## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.



Работа с инструментами модуля позволила понять разницу между подходом GitLab CI и подходом GitHub Actions. Самым простым оказалось написание кода ML-пайплайна. Основные трудности вызвала настройка авторизации через токены в среде Colab и синтаксические различия в структурах YAML-файлов.